Environment

In [ ]:
libs = {
    "hypernet": "/home/zanardi/Codes/TC/HyperNet/hypernet",
    "pycomet": "/home/zanardi/Codes/ML/PyCOMET/pycomet"
}
hypcfg = "./hypcfg.json"

In [ ]:
import sys
for lib in libs.values():
    sys.path.append(lib)
import os
os.environ.update({"HYPNET_CONFIGFILE": hypcfg})

Import libraries

In [ ]:
import json
import numpy as np
import tensorflow as tf

from scipy.linalg import expm
from silx.io.dictdump import h5todict

from hypernet import const
from hypernet.apps import utils as hyputils
from pycomet.networks import LinearMEGraphNet, NodePartitionFunctions, EdgeBwdEncoder, NodeTempEncoder

Plotting

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./figs/"
os.makedirs(path_to_figs, exist_ok=True)

def plot_var(
    x,
    y,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.show()
    plt.close()

Initializing thermophysical models

In [ ]:
with open("./../physics/inputs.json") as file:
    phyinp = json.load(file)
# Species thermo
sp_th = hyputils._get_specie_thermos(phyinp)
# Mixture
mix = hyputils._get_mixture(phyinp, sp_th)
# Kinetics
kin = hyputils._get_kinetics(phyinp, sp_th, mix)

Numerical solution

In [ ]:
sol = h5todict(h5file='./../physics/solution.hdf5')

In [ ]:
na, nm = [sol[f"X_{s}"] * sol['n'] for s in sp_th.keys()]

In [ ]:
# Compute source term
t, T = [sol[k] for k in ('t', 'T')]
n = np.hstack([na, nm])
omega = kin.function(t=t, y=[n, T])
dnmdt = omega[:,1:-1] / const.UNA
dnmdt = dnmdt.numpy().squeeze()

Analytical solution

In [ ]:
# Solution function
nm_int = lambda t, n0, k: np.matmul(expm(k*t), n0)

In [ ]:
# Construct the evolution matrix
kf, kb = 0.0, 0.0
for reaction in kin.reactions:
    kf = kf + reaction.fwd.k.numpy().squeeze()[0]
    kb = kb + reaction.bwd.k.numpy().squeeze()[0]
k = kb.T + kf
k = (k.T - np.diag(np.sum(k, axis=-1))) * na[0]
# Compute exact solution
nm_anal = np.vstack([nm_int(ti, nm[0], k) for ti in t])
# Compute source term
dnmdt_anal = np.einsum("ij,nj->ni",k,nm_anal) / const.UNA

In [ ]:
# plot_var(
#     x=t.squeeze()[1:],
#     y=nm[1:],
#     xref=t.squeeze()[1:],
#     yref=nm_anal[1:],
#     labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
#     scales=['log', 'linear'],
#     names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt.shape[1])],
#     figname=path_to_figs + "/solution"
# )

Construct graph

In [ ]:
# Nodes
node_attr = nm[...,None]
# Edges
k = kf + kb.T
k_idx = np.where(k > 0)
edge_index = np.array(k_idx)
edge_attr = (k[k_idx] * na[0]).reshape(1,-1,1)
edge_attr = np.tile(edge_attr, (len(node_attr),1,1))

In [ ]:
pf_lay1 = NodePartitionFunctions(
  temp_indices={
    "int": 1,
    "chem": 1,
    "trans": 1
  },
  specie_thermos={
    "filename": "/home/zanardi/Codes/ML/PyCOMET/pycomet/examples/gno/cg/physics/thermo/specie_thermos1.json",
    "thermo_process": 0,
    "eos_model": "PerfectGas",
    "config": None
  }
)
cfg1 = pf_lay1.get_config()

In [ ]:
pf_lay2 = NodePartitionFunctions(**cfg1)
cfg2 = pf_lay2.get_config()

In [ ]:
from deepdiff import DeepDiff
diff = DeepDiff(cfg1, cfg2)
diff

In [ ]:
indices = np.where(kf > 0)
bwd_lay = EdgeBwdEncoder(
    node_index_pf=[2],
    static_fwd_edge_index=np.vstack(indices),
    static_fwd_edge_attr=kf[indices]
)

In [ ]:
temp_attr = np.repeat(T, 3, axis=-1)[...,None]
x = np.concatenate([node_attr, temp_attr], -1)

In [ ]:
y = pf_lay1(x)
y = bwd_lay(y)

In [ ]:
tf.nest.map_structure(tf.shape, y)

In [ ]:
idx = tuple(np.split(y[1].numpy(), 2, axis=0))
rate = y[-1][0,:,0].numpy()
k = np.zeros((3,3))
k[idx] = rate
k

In [ ]:
kf + kb.T

In [ ]:
# Probability matrix
P = sp_th["O2"].specie.mappings["lev"].numpy()
# StS energies
ei = [sp_th["O2"].specie.lev[k].numpy() for k in ("ERot", "EVib")]
ei = np.stack(ei, axis=1) / const.eV_to_J
# CG energies
eg = (np.matmul(ei.T, P) / np.sum(P, axis=0, keepdims=True)).T

In [ ]:
inp_dim = 2 + 1             # Node energies (eg) and time (t)
temp_lay = NodeTempEncoder(
    layer_size=[inp_dim, 32, 32, 1],
    activation=["tanh"]*2 + ["linear"],
    kernel_initializer="GlorotNormal",
    regularizer={
        "name": "l1+l2",
        "l1": 1e-4,
        "l2": 1e-4
    },
    dropout={"rate": 1e-4},
    layer_norm=False,
    temp_lim=[1e2, 1e5],
    temp_xform="sigmoid",
    feat_indices=[0],
    static_node_attr=eg
)

In [ ]:
t_attr = np.tile(t[...,None], (1,3,1))
x = np.concatenate([t_attr, node_attr], -1)

In [ ]:
inp = tf.keras.Input(shape=tuple(x.shape[1:]))
out = temp_lay(inp)
model = tf.keras.Model(inp, out)

In [ ]:
y = model(x)
y.shape

Load linear master equation graph model

In [ ]:
model = LinearMEGraphNet(static_edge_index=edge_index)

In [ ]:
dnmdt_pred = model([node_attr, edge_attr]) / const.UNA
dnmdt_pred = dnmdt_pred.numpy().squeeze()

In [ ]:
plot_var(
    x=t.squeeze()[1:],
    y=dnmdt_pred[1:],
    xref=t.squeeze()[1:],
    yref=dnmdt[1:],
    labels=[r'$t$ [s]', r'$\omega$ [m$^{-3}$s$^{-1}$]'],
    scales=['log', 'linear'],
    names=[r'O$_2^{(%d)}$' % (i+1) for i in range(dnmdt.shape[1])],
    figname=path_to_figs + "/source"
)